# Answers: 3 · How the model learns

Worked answers to the **Try it yourself** exercises at the end of
[`3_how_the_model_learns.ipynb`](../3_how_the_model_learns.ipynb). Try each one yourself first. If you're stuck, run the cell here and
compare. Each answer runs on the same real data as the lesson, so you'll see real results,
not made-up ones.

The first code cell recreates just what the exercises need from the lesson, so this notebook
runs on its own.

In [ ]:
import sys; sys.path.insert(0, "..")
import io, contextlib
import numpy as np, pandas as pd, torch, torch.nn as nn
from tour import *
import surrogates

scores, embeddings = load_mini()
y = -scores["score"].to_numpy(np.float32)
order = np.random.default_rng(0).permutation(len(y))

def spearman(a, b):
    return pd.Series(a).corr(pd.Series(b), method="spearman")

def linear_model(n_train=420, weight_decay=1e-2, steps=300):
    # section 3's linear model, wrapped in a function so we can change its settings
    train, test = order[:n_train], order[n_train:]
    X = torch.tensor(embeddings["unimol2"])
    X = (X - X[train].mean(0)) / (X[train].std(0) + 1e-6)
    T = torch.tensor(y)
    torch.manual_seed(0)
    model = nn.Linear(768, 1)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=weight_decay)
    for _ in range(steps):
        opt.zero_grad()
        loss = ((model(X[train]).squeeze(1) - T[train]) ** 2).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (spearman(model(X[train]).squeeze(1).numpy(), y[train]),
                spearman(model(X[test]).squeeze(1).numpy(), y[test]))
print("ready")

---
## Exercise 1: `weight_decay=0`

In [ ]:
for wd in [1e-2, 0.0]:
    tr, te = linear_model(weight_decay=wd)
    print(f"weight_decay={wd:<5}  train {tr:.3f}   test {te:.3f}   gap {tr - te:.3f}")

**Answer.** Almost nothing changes. The gap between train and test Spearman is 0.078 with weight decay and 0.080 without. At this strength (`1e-2`, with only 300 Adam steps), weight decay has little effect: the model hasn't had time to memorize much either way. The lesson's point still holds, since the train score is always higher than the test score and only the test score is honest. But this setting doesn't show weight decay making a difference. To see it matter, train much longer or on fewer molecules. Measuring instead of assuming is the habit to keep.

---
## Exercise 2: fewer training molecules

The exercise asks about 100. We add a few more sizes to show the trend.

In [ ]:
for n in [50, 100, 420, 1000, 2000]:
    tr, te = linear_model(n_train=n)
    print(f"{n:5d} training molecules:  test Spearman {te:.3f}")

**Answer.** With 100 training molecules the test Spearman falls from 0.885 to 0.80, and with 50 to 0.71. More data keeps helping, with diminishing returns: 1,000 molecules gives 0.905 and 2,000 gives 0.914. The biggest gains come from the first few hundred molecules. That's good news for active learning: after just one or two rounds the model already orders the pool well.

---
## Exercise 3: `epochs=200` for the neural surrogate

In [ ]:
train, test = order[:420], order[420:]
rows = []
for name in BACKBONES:
    X = embeddings[name]
    for epochs in [50, 200]:
        torch.manual_seed(0)
        s = surrogates.SingleBackboneMVESurrogate(in_dim=X.shape[1])
        s.fit(X[train], y[train], epochs=epochs)
        rows.append({"backbone": name, "epochs": epochs,
                     "train Spearman": round(spearman(s.predict(X[train])[0], y[train]), 3),
                     "test Spearman": round(spearman(s.predict(X[test])[0], y[test]), 3)})
pd.DataFrame(rows)

**Answer.** It starts memorizing, slightly. With 200 epochs every backbone's **train** Spearman goes up (for example Uni-Mol2 0.942 → 0.977), but its **test** Spearman goes slightly down (0.876 → 0.865). MoLFormer and MHG-GED behave the same way. Training longer made the model better at the 420 molecules it saw and a little worse at new ones. That's overfitting, and it's why the project trains for a fixed, modest number of epochs. The ranking of the backbones doesn't change: MoLFormer stays far behind, whatever the training length.